# Xiangqi preprocessing: parse Kaggle dataset -> (board_tensor, score)

Notebook này xử lý dataset Xiangqi có 2 file:

- `gameinfo.csv`
- `move.csv`

Mục tiêu:

1. Đọc dữ liệu game metadata và move list
2. Tái dựng bàn cờ từ trạng thái ban đầu
3. Parse notation kiểu online Xiangqi như `C2.5`, `H2+3`, `R1.2`, `P7+1`
4. Sinh dữ liệu huấn luyện dạng `(board_tensor, score)`

`score` trong notebook này là weak label từ kết quả ván:

- red thắng -> `1.0`
- black thắng -> `-1.0`
- khác / hòa / không rõ -> `0.0`

Ta tạo sample theo kiểu:

- input = bàn cờ trước khi đi 1 nước
- label = kết quả cuối cùng của ván

Đây là cách đơn giản để chuẩn bị dữ liệu supervised ban đầu cho model value network.

## 1. Imports

In [77]:
import re
import json
import math
import copy
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
from typing import List, Tuple, Optional, Dict
import sys

## 2. Config

Sửa lại đường dẫn nếu cần.

In [78]:
# =========================
# Find project / ml / backend
# =========================
def find_repo_root():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / "backend" / "src").exists():
            return p
    raise FileNotFoundError("Không tìm thấy repo root chứa backend/src")

REPO_ROOT = find_repo_root()
PROJECT_DIR = Path.cwd().resolve()   # thường là thư mục đang mở notebook, ví dụ ml/
BACKEND_DIR = REPO_ROOT / "backend"
ML_DIR = BACKEND_DIR / "ml"

# Thêm backend vào sys.path để import được src.board, src.move_gen
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("Current working dir:", PROJECT_DIR)
print("REPO_ROOT         =", REPO_ROOT)
print("ML_DIR            =", ML_DIR)
print("BACKEND_DIR       =", BACKEND_DIR)
print("backend/src exists?", (BACKEND_DIR / "src").exists())

# =========================
# Dataset paths
# =========================
# Dataset của bạn đang ở: ml/dataset/online_chinese_chess_xiangqi_kaggle/
DATA_DIR = ML_DIR / "dataset" / "online_chinese_chess_xiangqi_kaggle"
GAMEINFO_CSV = DATA_DIR / "gameinfo.csv"
MOVE_CSV = DATA_DIR / "moves.csv"

# Output
OUTPUT_DIR = ML_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Preprocess settings
# =========================
MAX_GAMES = 1000
GENERATE_EVERY_PLY = True
TENSOR_MODE = "onehot"
INCLUDE_SIDE_TO_MOVE_PLANE = True

print("DATA_DIR      =", DATA_DIR)
print("GAMEINFO_CSV  =", GAMEINFO_CSV)
print("MOVE_CSV      =", MOVE_CSV)
print("OUTPUT_DIR    =", OUTPUT_DIR)
print("Exists gameinfo?", GAMEINFO_CSV.exists())
print("Exists moves?   ", MOVE_CSV.exists())

assert GAMEINFO_CSV.exists(), f"Không tìm thấy file: {GAMEINFO_CSV}"
assert MOVE_CSV.exists(), f"Không tìm thấy file: {MOVE_CSV}"

# =========================
# Import backend code
# =========================
from src.board import Board
from src.move_gen import MoveGenerator
from src.eval import Evaluator

print("Imported Board and MoveGenerator successfully.")

Current working dir: C:\co-tuong-ai\backend\ml
REPO_ROOT         = C:\co-tuong-ai
ML_DIR            = C:\co-tuong-ai\backend\ml
BACKEND_DIR       = C:\co-tuong-ai\backend
backend/src exists? True
DATA_DIR      = C:\co-tuong-ai\backend\ml\dataset\online_chinese_chess_xiangqi_kaggle
GAMEINFO_CSV  = C:\co-tuong-ai\backend\ml\dataset\online_chinese_chess_xiangqi_kaggle\gameinfo.csv
MOVE_CSV      = C:\co-tuong-ai\backend\ml\dataset\online_chinese_chess_xiangqi_kaggle\moves.csv
OUTPUT_DIR    = C:\co-tuong-ai\backend\ml\processed
Exists gameinfo? True
Exists moves?    True
Imported Board and MoveGenerator successfully.


## 3. Đọc dữ liệu CSV

In [79]:
gameinfo_df = pd.read_csv(GAMEINFO_CSV)
move_df = pd.read_csv(MOVE_CSV)

print('gameinfo shape:', gameinfo_df.shape)
print('move shape:', move_df.shape)
display(gameinfo_df.head())
display(move_df.head())

gameinfo shape: (10000, 7)
move shape: (672374, 4)


,gameID,game_datetime,blackID,blackELO,redID,redELO,winner
0,57380690,2017-02-10 13:11:10,levinhson,1382,baochang,1495,red
1,57380691,2017-02-10 13:21:43,phucnguyen,1269,quanem,1282,red
2,57380692,2017-02-10 13:23:04,pgm2785g,1253,swf2016g,1186,black
3,57380693,2017-02-10 13:09:58,cbk4833g,1209,ftc484g,1426,black
4,57380694,2017-02-10 13:18:06,stephenlee,919,tsk8201g,1235,red


,gameID,turn,side,move
0,57380690,1,red,C2.5
1,57380690,2,red,H2+3
2,57380690,3,red,R1.2
3,57380690,4,red,P7+1
4,57380690,5,red,C8.7


## 4. Chuẩn hóa dữ liệu

In [80]:
required_gameinfo_cols = ['gameID', 'winner']
required_move_cols = ['gameID', 'turn', 'side', 'move']

for c in required_gameinfo_cols:
    if c not in gameinfo_df.columns:
        raise ValueError(f'Missing column in gameinfo.csv: {c}')

for c in required_move_cols:
    if c not in move_df.columns:
        raise ValueError(f'Missing column in move.csv: {c}')

gameinfo_df = gameinfo_df.copy()
move_df = move_df.copy()

gameinfo_df['gameID'] = gameinfo_df['gameID'].astype(str)
move_df['gameID'] = move_df['gameID'].astype(str)
move_df['turn'] = pd.to_numeric(move_df['turn'], errors='coerce')
move_df['side'] = move_df['side'].astype(str).str.lower().str.strip()
move_df['move'] = move_df['move'].astype(str).str.strip()
gameinfo_df['winner'] = gameinfo_df['winner'].astype(str).str.lower().str.strip()

move_df = move_df.dropna(subset=['turn'])
move_df['turn'] = move_df['turn'].astype(int)
move_df = move_df.sort_values(['gameID', 'turn']).reset_index(drop=True)

if MAX_GAMES is not None:
    selected_ids = gameinfo_df['gameID'].drop_duplicates().astype(str).tolist()[:MAX_GAMES]
    gameinfo_df = gameinfo_df[gameinfo_df['gameID'].isin(selected_ids)].reset_index(drop=True)
    move_df = move_df[move_df['gameID'].isin(selected_ids)].reset_index(drop=True)

print('game count used:', gameinfo_df['gameID'].nunique())
print('move count used:', len(move_df))

game count used: 1000
move count used: 64612


## 5. Mapping quân cờ

Ta dùng quy ước số nguyên giống backend trong repo:

- Đỏ: số dương
- Đen: số âm
- Ô trống: `0`

Mã quân:

- King = 1
- Advisor = 2
- Elephant = 3
- Rook/Chariot = 4
- Horse = 5
- Cannon = 6
- Pawn = 7

In [81]:
EMPTY = 0

RED_KING = 1
RED_ADVISOR = 2
RED_ELEPHANT = 3
RED_ROOK = 4
RED_HORSE = 5
RED_CANNON = 6
RED_PAWN = 7

BLACK_KING = -1
BLACK_ADVISOR = -2
BLACK_ELEPHANT = -3
BLACK_ROOK = -4
BLACK_HORSE = -5
BLACK_CANNON = -6
BLACK_PAWN = -7

# Notation letters in dataset -> piece type id (absolute)
PIECE_LETTER_TO_ABS = {
    'K': 1, 'G': 1,  # king/general
    'A': 2,          # advisor
    'E': 3, 'B': 3, # elephant/bishop
    'R': 4,         # rook/chariot
    'H': 5, 'N': 5, # horse/knight
    'C': 6,         # cannon
    'P': 7,         # pawn
}

PIECE_TO_PLANE = {
    -7: 0,
    -6: 1,
    -5: 2,
    -4: 3,
    -3: 4,
    -2: 5,
    -1: 6,
     1: 7,
     2: 8,
     3: 9,
     4: 10,
     5: 11,
     6: 12,
     7: 13,
}

SIDE_TO_SIGN = {
    'red': 1,
    'black': -1,
}

def signed_piece(abs_piece: int, side: str) -> int:
    return abs_piece if side == 'red' else -abs_piece

## 6. Khởi tạo bàn cờ Xiangqi chuẩn

In [82]:
def initial_board() -> np.ndarray:
    board = np.zeros((10, 9), dtype=np.int8)

    # Black at top
    board[0] = [BLACK_ROOK, BLACK_HORSE, BLACK_ELEPHANT, BLACK_ADVISOR, BLACK_KING,
                BLACK_ADVISOR, BLACK_ELEPHANT, BLACK_HORSE, BLACK_ROOK]
    board[2, 1] = BLACK_CANNON
    board[2, 7] = BLACK_CANNON
    board[3, 0] = BLACK_PAWN
    board[3, 2] = BLACK_PAWN
    board[3, 4] = BLACK_PAWN
    board[3, 6] = BLACK_PAWN
    board[3, 8] = BLACK_PAWN

    # Red at bottom
    board[9] = [RED_ROOK, RED_HORSE, RED_ELEPHANT, RED_ADVISOR, RED_KING,
                RED_ADVISOR, RED_ELEPHANT, RED_HORSE, RED_ROOK]
    board[7, 1] = RED_CANNON
    board[7, 7] = RED_CANNON
    board[6, 0] = RED_PAWN
    board[6, 2] = RED_PAWN
    board[6, 4] = RED_PAWN
    board[6, 6] = RED_PAWN
    board[6, 8] = RED_PAWN
    return board

board0 = initial_board()
board0

array([[-4, -5, -3, -2, -1, -2, -3, -5, -4],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0, -6,  0,  0,  0,  0,  0, -6,  0],
       [-7,  0, -7,  0, -7,  0, -7,  0, -7],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 7,  0,  7,  0,  7,  0,  7,  0,  7],
       [ 0,  6,  0,  0,  0,  0,  0,  6,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 4,  5,  3,  2,  1,  2,  3,  5,  4]], dtype=int8)

## 7. Hàm hiển thị bàn cờ để debug

In [83]:
PIECE_TO_CHAR = {
    0: '.',
    1: 'K',  2: 'A',  3: 'E',  4: 'R',  5: 'H',  6: 'C',  7: 'P',
   -1: 'k', -2: 'a', -3: 'e', -4: 'r', -5: 'h', -6: 'c', -7: 'p',
}

def print_board(board: np.ndarray):
    print('   0 1 2 3 4 5 6 7 8')
    for r in range(10):
        row = ' '.join(PIECE_TO_CHAR[int(x)] for x in board[r])
        print(f'{r:2d} {row}')

print_board(board0)

   0 1 2 3 4 5 6 7 8
 0 r h e a k a e h r
 1 . . . . . . . . .
 2 . c . . . . . c .
 3 p . p . p . p . p
 4 . . . . . . . . .
 5 . . . . . . . . .
 6 P . P . P . P . P
 7 . C . . . . . C .
 8 . . . . . . . . .
 9 R H E A K A E H R


## 8. Tiện ích quy đổi cột theo notation

Trong Xiangqi notation, số cột thường tính từ phía người chơi đang đi:

- Red nhìn từ dưới lên: file 1..9 tương ứng cột từ trái sang phải của Red
- Black nhìn từ trên xuống: file 1..9 tương ứng cột từ phải sang trái theo hệ quy chiếu ma trận chuẩn

Với ma trận của ta:

- cột trái -> phải là `0..8`

Ta dùng quy ước practical sau:

- Red: file `1..9` -> col `8..0`
- Black: file `1..9` -> col `0..8`

Đây là một quy ước phổ biến khi bàn được lưu với Black ở trên, Red ở dưới.

In [84]:
def notation_file_to_col(file_num: int, side: str) -> int:
    assert 1 <= file_num <= 9
    if side == 'red':
        return 9 - file_num
    else:
        return file_num - 1

def col_to_notation_file(col: int, side: str) -> int:
    assert 0 <= col <= 8
    if side == 'red':
        return 9 - col
    else:
        return col + 1

## 9. Luật sinh nước đi hợp lệ cơ bản

Để parse notation đáng tin cậy, ta cần biết quân nào thực sự có thể đi tới ô đích.

Tái sử dụng board.py

In [85]:
def initial_backend_board():
    return Board()

def board_to_numpy(board_obj: Board) -> np.ndarray:
    return np.array(board_obj.board, dtype=np.int8)

def clone_board(board_obj: Board) -> Board:
    b = Board()
    b.board = [row[:] for row in board_obj.board]
    b.current_player = board_obj.current_player
    return b

In [86]:
def clone_board(board_obj: Board) -> Board:
    b = Board()
    b.board = [row[:] for row in board_obj.board]
    b.current_player = board_obj.current_player
    return b

def board_to_numpy(board_obj: Board):
    import numpy as np
    return np.array(board_obj.board, dtype=np.int8)

def piece_can_move_with_backend(board_obj: Board, r1: int, c1: int, r2: int, c2: int) -> bool:
    piece = board_obj.get_piece(r1, c1)
    if piece == Board.EMPTY:
        return False
    move_gen = MoveGenerator(board_obj)
    legal_moves = move_gen.generate_moves(r1, c1)
    return (r2, c2) in legal_moves

## 10. Parse notation move

Ta giả định move có dạng cơ bản:

- `<Piece><from_file><op><arg>`

Ví dụ:

- `C2.5`
- `H2+3`
- `R1.2`
- `P7+1`

Giải thích practical:

- `Piece`: loại quân
- `from_file`: cột xuất phát theo góc nhìn bên đi
- `op`:
  - `.` = đi ngang tới file đích
  - `+` = tiến
  - `-` = lùi
- `arg`:
  - với rook/cannon/king/pawn: thường là số bước nếu `+/-`, hoặc file đích nếu `.`
  - với horse/elephant/advisor: thường là file đích đối với `+/-`

Notebook dùng chiến lược robust:

1. parse chuỗi notation
2. tìm các quân cùng loại nằm trên `from_file`
3. từ đó suy ra ô đích theo notation
4. validate bằng `piece_can_move`

Nếu nhiều ứng viên, chọn ứng viên hợp lệ duy nhất. Nếu vẫn mơ hồ, báo fail.

In [87]:
MOVE_RE = re.compile(r'^([A-Za-z])([1-9])([+\-.])([1-9])$')

def parse_move_string(move_str: str):
    m = MOVE_RE.match(move_str.strip())
    if not m:
        raise ValueError(f'Unsupported move format: {move_str}')
    piece_letter, from_file, op, arg = m.groups()
    piece_letter = piece_letter.upper()
    from_file = int(from_file)
    arg = int(arg)
    if piece_letter not in PIECE_LETTER_TO_ABS:
        raise ValueError(f'Unknown piece letter: {piece_letter} in move {move_str}')
    return {
        'piece_letter': piece_letter,
        'abs_piece': PIECE_LETTER_TO_ABS[piece_letter],
        'from_file': from_file,
        'op': op,
        'arg': arg,
    }

def find_piece_candidates(board: np.ndarray, side: str, abs_piece: int, from_file: int):
    target_piece = signed_piece(abs_piece, side)
    target_col = notation_file_to_col(from_file, side)
    candidates = []
    for r in range(10):
        if board[r, target_col] == target_piece:
            candidates.append((r, target_col))
    return candidates

def forward_dir(side: str) -> int:
    # red goes up -> -1, black goes down -> +1
    return -1 if side == 'red' else 1

def infer_destinations_from_notation(board: np.ndarray, side: str, src: Tuple[int, int], parsed: Dict):
    r, c = src
    abs_piece = parsed['abs_piece']
    op = parsed['op']
    arg = parsed['arg']
    dests = []

    # Horizontal move: destination file
    if op == '.':
        c2 = notation_file_to_col(arg, side)
        # row remains same for orthogonal horizontal pieces.
        # For horse/elephant/advisor, dot notation is uncommon; skip unless valid later.
        for r2 in range(10):
            if abs_piece in {4, 6, 1, 7}:
                r2 = r
                dests.append((r2, c2))
                break
            else:
                dests.append((r2, c2))
        return list(dict.fromkeys(dests))

    # Forward/backward
    step_sign = 1 if op == '+' else -1
    # Convert relative notation to matrix row delta
    # '+' means move forward from player's perspective
    # '-' means move backward from player's perspective
    fd = forward_dir(side)
    row_dir = fd * step_sign

    # Rook / Cannon / King / Pawn: arg usually steps
    if abs_piece in {1, 4, 6, 7}:
        r2 = r + row_dir * arg
        c2 = c
        dests.append((r2, c2))

        # Some notation variants may use target file for king/pawn in odd cases.
        # Keep fallback candidates conservative.
        return dests

    # Horse: arg is usually destination file
    if abs_piece == 5:
        c2 = notation_file_to_col(arg, side)
        # horse changes row by 1 or 2 depending on column delta
        for dr in [1, 2, -1, -2]:
            r2 = r + dr
            dests.append((r2, c2))
        # prioritize forward direction later using legality + row_dir preference
        return list(dict.fromkeys(dests))

    # Elephant / Advisor: arg usually destination file
    if abs_piece in {2, 3}:
        c2 = notation_file_to_col(arg, side)
        for dr in [1, -1, 2, -2]:
            r2 = r + dr
            dests.append((r2, c2))
        return list(dict.fromkeys(dests))

    return dests

def choose_best_destination(board_obj: Board, side: str, src, parsed, dests):
    op = parsed['op']
    abs_piece = parsed['abs_piece']
    r, c = src

    legal = []
    for r2, c2 in dests:
        if not (0 <= r2 < 10 and 0 <= c2 < 9):
            continue
        if piece_can_move_with_backend(board_obj, r, c, r2, c2):
            legal.append((r2, c2))

    if not legal:
        return None

    if len(legal) == 1:
        return legal[0]

    # heuristic giữ nguyên như trước
    if op in ['+', '-']:
        fd = -1 if side == 'red' else 1
        want_sign = fd if op == '+' else -fd
        filtered = []
        for r2, c2 in legal:
            dr = r2 - r
            if dr == 0:
                continue
            if (dr > 0 and want_sign > 0) or (dr < 0 and want_sign < 0):
                filtered.append((r2, c2))
        if len(filtered) == 1:
            return filtered[0]
        if filtered:
            legal = filtered

    return legal[0]

def resolve_move(board_obj: Board, side: str, move_str: str):
    parsed = parse_move_string(move_str)
    board_np = np.array(board_obj.board)

    candidates = find_piece_candidates(board_np, side, parsed['abs_piece'], parsed['from_file'])
    if not candidates:
        raise ValueError(f'No source piece found for move={move_str}, side={side}')

    resolved = []
    for src in candidates:
        dests = infer_destinations_from_notation(board_np, side, src, parsed)
        dst = choose_best_destination(board_obj, side, src, parsed, dests)
        if dst is not None:
            resolved.append((src, dst))

    if not resolved:
        raise ValueError(f'Cannot resolve move={move_str}, side={side}, candidates={candidates}')

    if len(resolved) > 1:
        if side == 'red':
            resolved = sorted(resolved, key=lambda x: x[0][0], reverse=True)
        else:
            resolved = sorted(resolved, key=lambda x: x[0][0])

    return resolved[0]

def apply_move_backend(board_obj: Board, src, dst) -> Board:
    new_board = clone_board(board_obj)
    ok = new_board.move_piece(src[0], src[1], dst[0], dst[1])
    if not ok:
        raise ValueError(f"Illegal backend move: {src} -> {dst}")
    return new_board

## 11. Test parser với vài move đơn giản

In [88]:
test_board = initial_board()
print('Initial board:')
print_board(test_board)

test_moves = [
    ('red', 'C2.5'),
    ('red', 'H2+3'),
    ('red', 'R1.2'),
    ('red', 'P7+1'),
]

for side, mv in test_moves:
    try:
        src, dst = resolve_move(test_board, side, mv)
        print(f'{side} {mv}: {src} -> {dst}')
    except Exception as e:
        print(f'FAILED {side} {mv}: {e}')

Initial board:
   0 1 2 3 4 5 6 7 8
 0 r h e a k a e h r
 1 . . . . . . . . .
 2 . c . . . . . c .
 3 p . p . p . p . p
 4 . . . . . . . . .
 5 . . . . . . . . .
 6 P . P . P . P . P
 7 . C . . . . . C .
 8 . . . . . . . . .
 9 R H E A K A E H R
FAILED red C2.5: 'numpy.ndarray' object has no attribute 'board'
FAILED red H2+3: 'numpy.ndarray' object has no attribute 'board'
FAILED red R1.2: 'numpy.ndarray' object has no attribute 'board'
FAILED red P7+1: 'numpy.ndarray' object has no attribute 'board'


## 12. Encoding board -> tensor

In [89]:
def board_to_int_tensor(board: np.ndarray) -> np.ndarray:
    return board.astype(np.int8)

def board_to_onehot_tensor(board: np.ndarray, side_to_move: Optional[str] = None, include_side_plane: bool = True) -> np.ndarray:
    num_planes = 14 + (1 if include_side_plane else 0)
    x = np.zeros((num_planes, 10, 9), dtype=np.float32)

    for r in range(10):
        for c in range(9):
            piece = int(board[r, c])
            if piece == 0:
                continue
            plane = PIECE_TO_PLANE[piece]
            x[plane, r, c] = 1.0

    if include_side_plane:
        plane_idx = 14
        if side_to_move == 'red':
            x[plane_idx, :, :] = 1.0
        else:
            x[plane_idx, :, :] = 0.0

    return x

def encode_board(board: np.ndarray, side_to_move: Optional[str] = None, mode: str = 'onehot') -> np.ndarray:
    if mode == 'int':
        return board_to_int_tensor(board)
    elif mode == 'onehot':
        return board_to_onehot_tensor(board, side_to_move=side_to_move, include_side_plane=INCLUDE_SIDE_TO_MOVE_PLANE)
    else:
        raise ValueError(f'Unknown mode: {mode}')

## 13. Winner -> score

In [90]:
def get_heuristic_score(board_obj: Board) -> float:
    evaluator = Evaluator(board_obj)
    return float(evaluator.evaluate())

Normalize score

In [91]:
def normalize_score(score: float, scale: float = 300.0) -> float:
    return float(np.tanh(score / scale))

## 14. Tạo samples từ 1 game

Mỗi sample gồm:

- `board_tensor`
- `score`
- metadata phụ như `gameID`, `turn`, `side`, `move`

Ta encode trạng thái **trước khi đi nước đó**.

In [92]:
def sort_game_moves(game_moves: pd.DataFrame) -> pd.DataFrame:
    side_order = {'red': 0, 'black': 1}
    gm = game_moves.copy()
    gm['side_order'] = gm['side'].map(side_order)
    gm = gm.sort_values(['turn', 'side_order']).reset_index(drop=True)
    return gm.drop(columns=['side_order'])

In [93]:
def build_samples_for_game(game_id: str, game_moves: pd.DataFrame, tensor_mode='onehot'):
    board_obj = Board()
    samples = []
    errors = []

    game_moves = game_moves = sort_game_moves(game_moves)

    for _, row in game_moves.iterrows():
        turn = int(row['turn'])
        side = str(row['side']).lower().strip()
        move_str = str(row['move']).strip()

        if side not in ('red', 'black'):
            errors.append({
                'gameID': game_id,
                'turn': turn,
                'move': move_str,
                'error': f'invalid side {side}'
            })
            continue

        try:
            # 1. Encode trạng thái TRƯỚC nước đi
            board_np = np.array(board_obj.board, dtype=np.int8)
            board_tensor = encode_board(board_np, side_to_move=side, mode=tensor_mode)

            # 2. Lấy heuristic score
            raw_score = float(Evaluator(board_obj).evaluate())
            score = float(np.tanh(raw_score / 300.0))

            samples.append({
                'gameID': game_id,
                'turn': turn,
                'side': side,
                'move': move_str,
                'board_tensor': board_tensor,
                'raw_score': raw_score,
                'score': np.float32(score),
            })

            # 3. Resolve move rồi apply move
            src, dst = resolve_move(board_obj, side, move_str)
            board_obj = apply_move_backend(board_obj, src, dst)

        except Exception as e:
            errors.append({
                'gameID': game_id,
                'turn': turn,
                'side': side,
                'move': move_str,
                'error': str(e),
            })
            break

    return samples, errors

## 15. Chạy preprocessing toàn bộ game

In [94]:
moves_by_game = {gid: df for gid, df in move_df.groupby('gameID')}

all_samples = []
all_errors = []

for _, row in gameinfo_df.iterrows():
    game_id = str(row['gameID'])
    gm = moves_by_game.get(game_id)

    if gm is None or len(gm) == 0:
        continue

    samples, errors = build_samples_for_game(game_id, gm, tensor_mode=TENSOR_MODE)
    all_samples.extend(samples)
    all_errors.extend(errors)

print('num samples:', len(all_samples))
print('num errors:', len(all_errors))

if all_errors:
    display(pd.DataFrame(all_errors).head(20))

num samples: 2200
num errors: 984


,gameID,turn,side,move,error
0,57380690,1,black,h2+3,"Cannot resolve move=h2+3, side=black, candidat..."
1,57380691,1,black,h2+3,"Cannot resolve move=h2+3, side=black, candidat..."
2,57380692,1,black,h8+7,"Cannot resolve move=h8+7, side=black, candidat..."
3,57380693,1,red,E3+5,"Cannot resolve move=E3+5, side=red, candidates..."
4,57380694,1,black,h2+3,"Cannot resolve move=h2+3, side=black, candidat..."
5,57380695,1,red,H2+3,"Cannot resolve move=H2+3, side=red, candidates..."
6,57380696,1,red,H8+7,"Cannot resolve move=H8+7, side=red, candidates..."
7,57380697,2,red,H2+3,"Cannot resolve move=H2+3, side=red, candidates..."
8,57380698,1,black,h2+3,"Cannot resolve move=h2+3, side=black, candidat..."
9,57380699,1,black,e3+5,"Cannot resolve move=e3+5, side=black, candidat..."


## 16. Kiểm tra shape tensor

In [95]:
if all_samples:
    s0 = all_samples[0]
    print('sample keys:', s0.keys())
    print('board_tensor shape:', s0['board_tensor'].shape)
    print('score:', s0['score'])
    print('move:', s0['move'])
    print('side:', s0['side'])

sample keys: dict_keys(['gameID', 'turn', 'side', 'move', 'board_tensor', 'raw_score', 'score'])
board_tensor shape: (15, 10, 9)
score: 0.0
move: C2.5
side: red


## 17. Gom thành numpy arrays

Ta tách:

- `X`: mảng tensor bàn cờ
- `y`: mảng score
- metadata phụ

In [96]:
X = np.stack([s['board_tensor'] for s in all_samples], axis=0) if all_samples else np.array([])
y = np.array([s['score'] for s in all_samples], dtype=np.float32) if all_samples else np.array([])

meta_df = pd.DataFrame([
    {
        'gameID': s['gameID'],
        'turn': s['turn'],
        'side': s['side'],
        'move': s['move'],
        'score': float(s['score']),
    }
    for s in all_samples
])

print('X shape:', X.shape)
print('y shape:', y.shape)
display(meta_df.head())

X shape: (2200, 15, 10, 9)
y shape: (2200,)


,gameID,turn,side,move,score
0,57380690,1,red,C2.5,0.0
1,57380690,1,black,h2+3,0.0
2,57380691,1,red,C2.5,0.0
3,57380691,1,black,h2+3,0.0
4,57380692,1,red,C8.6,0.0


## 18. Lưu output

Ta lưu:

- `xiangqi_X.npy`
- `xiangqi_y.npy`
- `xiangqi_meta.csv`
- `errors.csv`

Nếu muốn gọn hơn thì lưu `.npz`.

In [97]:
# Chọn thư mục output riêng cho heuristic label
SAVE_DIR = OUTPUT_DIR / "heuristic_label"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

x_path = SAVE_DIR / "xiangqi_X.npy"
y_path = SAVE_DIR / "xiangqi_y.npy"
meta_path = SAVE_DIR / "xiangqi_meta.csv"
err_path = SAVE_DIR / "errors.csv"
npz_path = SAVE_DIR / "xiangqi_dataset.npz"

if len(all_samples) > 0:
    np.save(x_path, X)
    np.save(y_path, y)
    meta_df.to_csv(meta_path, index=False)
    np.savez_compressed(npz_path, X=X, y=y)

pd.DataFrame(all_errors).to_csv(err_path, index=False)

print("saved:")
print("-", x_path)
print("-", y_path)
print("-", meta_path)
print("-", err_path)
print("-", npz_path)

saved:
- C:\co-tuong-ai\backend\ml\processed\heuristic_label\xiangqi_X.npy
- C:\co-tuong-ai\backend\ml\processed\heuristic_label\xiangqi_y.npy
- C:\co-tuong-ai\backend\ml\processed\heuristic_label\xiangqi_meta.csv
- C:\co-tuong-ai\backend\ml\processed\heuristic_label\errors.csv
- C:\co-tuong-ai\backend\ml\processed\heuristic_label\xiangqi_dataset.npz


## 19. Cách dùng output cho training

Ví dụ:

- nếu `TENSOR_MODE='int'` thì `X.shape = (N, 10, 9)`
- nếu `TENSOR_MODE='onehot'` và có side plane thì `X.shape = (N, 15, 10, 9)`

`y.shape = (N,)` với giá trị trong `{-1, 0, 1}`.

In [98]:
if X.size > 0:
    print('Example X dtype:', X.dtype)
    print('Example y unique:', np.unique(y))
    print('First sample metadata:')
    display(meta_df.iloc[:5])

Example X dtype: float32
Example y unique: [-0.527494   -0.2851999   0.          0.06656808  0.07320216]
First sample metadata:


,gameID,turn,side,move,score
0,57380690,1,red,C2.5,0.0
1,57380690,1,black,h2+3,0.0
2,57380691,1,red,C2.5,0.0
3,57380691,1,black,h2+3,0.0
4,57380692,1,red,C8.6,0.0


## 20. Ghi chú quan trọng

### A. Vì sao `score` lấy từ `winner`?

Vì dataset hiện tại chỉ có:

- thông tin ván (`winner`, ELO, ...)
- chuỗi nước đi

Nó **không có engine evaluation score cho từng position**. Vì vậy cách dễ nhất là dùng kết quả cuối ván làm nhãn yếu (weak label).

### B. `board_tensor` nên chọn dạng nào?

- `int (10x9)`: đơn giản, tiết kiệm bộ nhớ
- `onehot (14/15 planes)`: tốt hơn cho CNN

### C. Parser notation có thể fail ở đâu?

Dataset Xiangqi online đôi khi có biến thể notation, hoặc notation rút gọn khi nhiều quân cùng cột. Notebook này hỗ trợ format cơ bản như ví dụ bạn đưa, nhưng với dataset thật lớn có thể còn một tỉ lệ nhỏ move không parse được.

### D. Nếu muốn tốt hơn

Bạn có thể nâng cấp thêm:

1. xử lý notation mơ hồ khi 2 quân cùng loại cùng file
2. bổ sung kiểm tra flying-general đầy đủ
3. dùng Stockfish/Pikafish/Xiangqi engine để sinh `score` thật cho từng position
4. sinh thêm target policy từ nước đi thực tế